# Assignment 8 - Group 1

**Group members** :
- Max Chipani
- Jesus Gamboa
- Karen Salazar
- Paolo Gutierrez
- Luis Camarena

In [ ]:
#The following command installs the geopandas, rasterio and rasterstats libraries.
!pip install geopandas rasterio rasterstats

In [ ]:
# Import bookstores
import geopandas as gpd #Import the GeoPandas library to work with geospatial data in vector format (points, lines, polygons).
import pandas as pd # Import pandas for data manipulation in the form of tables (DataFrames).
import numpy as np # Import NumPy for numerical operations and array handling.
import matplotlib.pyplot as plt # Import Matplotlib to create graphs and visualizations.
import rasterio # Import raster to work with raster data (georeferenced images).
import pyproj # paraImport pyproj for geographic coordinate projection.
import glob # Import glob find files that match a specific pattern on the file system.
# Importa Folium para crear mapas interactivos basados en Leaflet.js.
import folium as fm # Import Folium to create interactive maps based on Leaflet.js.
from rasterstats import zonal_stats # Import the zonal_stats function from rasterstats to calculate zonal statistics between raster and vector data.
from shapely.ops import transform # Import shapely's transform function to transform geometries, such as changing the coordinate system.

## Question 1

Get the 15 variables from this raster for all Peru departments polygons. This is the link where shapefiles are located. This is the link of the source raster. The values should be the percentage of district area cover by this specific Morphological Settlement Zone.

In [ ]:
# Read shapefile containing district boundaries 
dist = gpd.read_file("../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp")

# Select relevant columns
dist = dist[['CCDD', 'NOMBDEP', 'NOMBDIST', 'UBIGEO', 'geometry']]

# Display the resulting GeoDataFrame with the selected columns
dist

In [ ]:
# Set up a transformer to convert coordinates from WGS84 to Mollweide projection
transformer = pyproj.Transformer.from_crs('epsg:4326', 'esri:54009', always_xy=True)

# Define a function to apply the coordinate transformation to geometries
def apply_transform(geom):
    return transform(transformer.transform, geom)

# Apply the transformation to the districts geometries
dist['geometry'] = dist['geometry'].apply(apply_transform)

In [ ]:
# Get a list of all .tif raster files in the current directory
lista_rasters = glob.glob("*.tif")
lista_rasters

In [ ]:
x = []

# Obtain the values counts in each raster with district geometry
for raster in lista_rasters:
    print(raster)
    stats = zonal_stats(dist, raster,
                       stats='unique',
                       categorical=True)
    df1 = pd.DataFrame(stats)
    df1['UBIGEO'] = dist['UBIGEO']
    x.append(df1)

In [ ]:
# Create the value counts of each variable at district level
df_ubigeo = pd.concat(x).replace(np.nan, 0).groupby('UBIGEO').sum().reset_index()
df_ubigeo

In [ ]:
# Create the value counts of each variable at departament level
df_final = pd.merge(dist, df_ubigeo, on='UBIGEO',  how='left')
df_final = df_final.drop(['UBIGEO', 'geometry'], axis=1)
df_final = df_final.groupby(['CCDD', 'NOMBDEP']).sum()
df_final = df_final.reset_index()
df_final = df_final.drop(['NOMBDIST', 'unique', 0], axis=1)
df_final

In [ ]:
# Calculate the percentage of district area cover by this specific Morphological Settlement Zone.
variables = df_final.columns[2:]
for var in variables:
    df_final[f'MSZ_{var}'] = df_final[var] / df_final[var].sum() * 100

df_final = df_final[['CCDD', 'NOMBDEP'] + [f'MSZ_{var}' for var in variables]]
df_final = df_final.replace(np.nan, 0)
df_final

## Question 2

Then you are going to generate choropleth map using folium for these 15 variables.

In [ ]:
# Load the shapefile with department boundaries using GeoPandas
depa = gpd.read_file("../../_data/INEI_LIMITE_DEPARTAMENTAL/INEI_LIMITE_DEPARTAMENTAL.shp")
# Keep only 'CCDD' (department code) and 'geometry' (boundaries)
depa = depa[['CCDD', 'geometry']]
# Simplify the geometries to reduce detail with a tolerance of 0.01
depa['geometry'] = depa['geometry'].simplify(tolerance=0.01)

In [ ]:
# Latitude and longitude of the palace location
lat_palacio = -12.0757538
long_palacio = -76.9863174
# Create a map centered at the palace with an initial zoom level of 5
map = fm.Map(location=[lat_palacio, long_palacio], zoom_start=5)

In [ ]:
# Select columns from the third onward, excluding 'geometry'
variables = df_final.columns[2:]

# Loop through each variable to create a choropleth map
for var in variables:
    if var != 'geometry':  # Skip the 'geometry' column
        fm.Choropleth(
            geo_data=depa,  # Use department boundaries
            name=var,  # Name each layer after the variable
            data=df_final,  # Data source
            columns=['CCDD', var],  # Data columns: department code and the variable
            key_on='feature.properties.CCDD',  # Match on department code
            fill_color='YlGn',  # Use 'Yellow-Green' color scale
            fill_opacity=0.7,  # Set transparency of the fill color
            line_opacity=0.2,  # Set transparency of the boundaries
            legend_name=var  # Display the variable as the legend
        ).add_to(map)

# Add layer control to toggle between choropleth layers
fm.LayerControl().add_to(map)


## Question 3

Save your html in the same folder of your JN. Name your HTML as your branch. This HTML should have all these layers. Please do not forget to use Layer Control.



In [ ]:
# Saves the map object to an HTML file with the specified filename ('group_1_ass_8_2024_2.html').
map.save('group_1_ass_8_2024_2.html')

In [ ]:
map  # Displays the map object in the current environment, such as a Jupyter notebook or interactive environment.

**Legend:**

01 : MSZ, open spaces, low vegetation surfaces NDVI <= 0.3

02 : MSZ, open spaces, medium vegetation surfaces 0.3 < NDVI <=0.5

03 : MSZ, open spaces, high vegetation surfaces NDVI > 0.5

04 : MSZ, open spaces, water surfaces LAND < 0.5

05 : MSZ, open spaces, road surfaces

11 : MSZ, built spaces, residential, building height <= 3m

12 : MSZ, built spaces, residential, 3m < building height <= 6m

13 : MSZ, built spaces, residential, 6m < building height <= 15m

14 : MSZ, built spaces, residential, 15m < building height <= 30m

15 : MSZ, built spaces, residential, building height > 30m

21 : MSZ, built spaces, non-residential, building height <= 3m

22 : MSZ, built spaces, non-residential, 3m < building height <= 6m

23 : MSZ, built spaces, non-residential, 6m < building height <= 15m

24 : MSZ, built spaces, non-residential, 15m < building height <= 30m

25 : MSZ, built spaces, non-residential, building height > 30m